In [ ]:
# apt update -y
# apt install poppler-utils tesseract-ocr tesseract-ocr-vie -y
# pip install -U pytesseract pdf2image pdfplumber pillow sentence-transformers pandas PyMuPDF


SyntaxError: invalid syntax (3593776737.py, line 1)

In [ ]:
import os
import json
import pytesseract
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
import fitz  # PyMuPDF
import random
import multiprocessing as mp


folder_path = "test_data"   # thư mục chứa PDF
docs = []


def ocr_image(img):
    """Worker OCR 1 ảnh (nhanh hơn ocr_page cũ)"""
    return pytesseract.image_to_string(img, lang="vie").strip()


def read_pdf_smart(file_path):
    pdf = fitz.open(file_path)
    total_pages = len(pdf)

    sample_pages = random.sample(range(total_pages), k=min(5, total_pages))
    text_count = sum(
        1 for p in sample_pages if pdf.load_page(p).get_text("text").strip()
    )

    # CASE 1 → PDF có text
    if text_count >= 3:
        print("✅ Dùng PyMuPDF TEXT cho:", os.path.basename(file_path))
        texts = [
            page.get_text("text").strip()
            for page in pdf
            if page.get_text("text").strip()
        ]
        pdf.close()
        return texts

    # CASE 2 → OCR
    print("⚠️ PDF scan → OCR multiprocess:", os.path.basename(file_path))

    try:
        # thử dùng poppler
        images = convert_from_path(file_path, dpi=200, thread_count=4)
    except Exception:
        print("❌ Poppler ERROR → fallback PyMuPDF export")
        images = []
        for i in range(total_pages):
            pix = pdf.load_page(i).get_pixmap(dpi=200)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(img)

    pdf.close()

    cpu_use = max(1, mp.cpu_count() - 1)
    print(f"🔧 Số CPU sử dụng: {cpu_use}")

    with mp.Pool(cpu_use) as pool:
        results = pool.map(ocr_image, images)

    results = [t for t in results if t]

    return results

chunk_id = 0

for file_name in os.listdir(folder_path):
    if file_name.lower().endswith(".pdf"):
        file_path = os.path.join(folder_path, file_name)

        all_texts = read_pdf_smart(file_path)

        for text in all_texts:
            docs.append({
                "chunk_id": chunk_id,
                "text": text,
                "file_path": file_path,
                "file_name": file_name
            })
            chunk_id += 1

print("✅ Tổng số chunk:", len(docs))
print("✅ 3 chunk đầu:")
print(docs[:3])


✅ Dùng PyMuPDF TEXT cho: EBOOK 150 DẪN CHỨNG NLXH VỀ MỘT TƯ TƯỞNG ĐẠO LÝ.pdf
⚠️ PDF scan → OCR multiprocess: EBOOK CHINH PHỤC HOÁ VÔ CƠ LỚP 12 THẦY PHẠM VĂN TRỌNG.pdf
🔧 Số CPU sử dụng: 11


In [ ]:
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

rows = []

for d in docs:
    vec = embedder.encode(d["text"]).tolist()

    rows.append({
        "chunk_id": d["chunk_id"],
        "text": d["text"],
        "embedding": json.dumps(vec),
        "file_path": d["file_path"],
        "file_name": d["file_name"]
    })

df = pd.DataFrame(rows)
df.to_csv("vector_store.csv", index=False)

print("✅ Đã lưu vector_store.csv")
